# **BIBLIOTECAS, FUNÇÕES E APIs**

In [5]:
!pip install yfinance

     ---------------------------------------- 0.0/3.0 MB ? eta -:--:--
     ---------------------------------------- 3.0/3.0 MB 17.7 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 17.2 MB/s eta 0:00:00
  Created wheel for peewee: filename=peewee-3.18.1-py3-none-any.whl size=139097 sha256=6f683eb045d531b16c84e6f507d3499a81f822efd6877109767aec12ccadb62f
  Stored in directory: c:\users\beatr\appdata\local\pip\cache\wheels\e7\09\87\1f1a87eb25af98d0a1f60ef184a9779d81c6a2e8d28bc67d7f
Successfully built peewee


In [7]:
import pandas as pd
import requests
import yfinance as yf


In [24]:
#Função de limpeza e transposição da tabela

def clean_pd_df(df, primeiro_indice, segundo_indice, valor, valory ):
    try:
        # Encontrar o índice da linha contendo 'x'
        indice_x = df.index[df.iloc[:, 0].astype(str) ==  primeiro_indice].tolist()[0] + valor
        # # Encontrar o índice da linha contendo 'y'
        indice_y = df.index[df.iloc[:, 0].astype(str) == segundo_indice].tolist()[0] + valory
        # Selecionar o intervalo de linhas entre 'x' e 'y'
        df_clean = (df.iloc[indice_x :indice_y, :])
        # Filtra linhas onde todos os valores são NaN
        df_filtrado = df_clean.dropna(how='all')
        #Exclui linhas que contém Total
        df_filtrado[~df_filtrado.iloc[:, 0].astype(str).str.contains('Total', case=False, na=False)].reset_index(drop=True)
        #Filtra colunas onde todos os valores são NaN
        df = df_filtrado.dropna(axis=1, how='all')

        #Indice de Coluna 'z'

        return df
    except IndexError:
        print("Não foi possível encontrar os índices")
        return None

def transpor_tabela(df):
    df.columns = df.iloc[0].astype(str)
    df = df[1:].reset_index(drop=True)
    df.rename(columns={df.columns[0]: "Country"}, inplace=True)

    return df

# **ENERGY INST**

In [26]:
#Encontra o nome das abas da planilha
caminho_do_arquivo = 'ARQUIVOS/ENERGYINST/EI-Stats-Review-All-Data.xlsx'

try:
    excel_file = pd.ExcelFile(caminho_do_arquivo)

    # Obtém a lista de nomes das planilhas
    nomes_das_planilhas = excel_file.sheet_names

    # Converte para uma Series para aplicar o filtro com str.contains
    nomes_series = pd.Series(nomes_das_planilhas)
    nomes_filtrados = nomes_series[nomes_series.str.contains('Oil', case=True, na=False)]

    print(nomes_filtrados)

except Exception as e:
    print("Erro:", e)

11              Oil - Proved reserves
12      Oil - Proved reserves history
13           Oil Production - barrels
14            Oil Production - tonnes
18          Oil Consumption - barrels
19           Oil Consumption - Tonnes
20               Oil Consumption - EJ
21         Oil - Regional Consumption
23        Oil crude prices since 1861
24          Oil refinery - throughput
25            Oil refinery - capacity
26    Oil - Regional refining margins
27                Oil trade movements
28         Oil - Inter-area movements
29     Oil - Trade movements in 22-23
77      Oil inputs - Elec generation 
dtype: object


In [53]:
#Historico de Reservas de Petroleo -
historico_reservas = pd.read_excel('ARQUIVOS/ENERGYINST/EI-Stats-Review-All-Data.xlsx', sheet_name = 'Oil - Proved reserves history', header=None)
historico_reservas_clean = transpor_tabela(clean_pd_df(historico_reservas, 'Thousand million barrels', 'Total World', 0, 0 ))
historico_reservas = pd.melt(historico_reservas_clean, id_vars=["Country"], var_name="Year", value_name= 'Reserva')
historico_reservas['Year'] = historico_reservas['Year'].str.extract(r'^(\d+)').astype(int)


# # #Historico de Consumo de Petroleo -
producao_barril = pd.read_excel('ARQUIVOS/ENERGYINST/EI-Stats-Review-All-Data.xlsx', sheet_name = 'Oil Consumption - barrels', header=None)
producao_barril_clean = transpor_tabela(clean_pd_df(producao_barril, 'Thousand barrels daily', 'Total World', 0, 0 ))
producao_barril = pd.melt(producao_barril_clean, id_vars=["Country"], var_name="Year", value_name= 'Consuption')
producao_barril['Year'] = producao_barril['Year'].str.extract(r'^(\d+)').astype(int)
producao_barril = producao_barril[producao_barril['Year'] >= 1980]


# #Historico de Produção de Petroleo -
consumo_barril = pd.read_excel('ARQUIVOS/ENERGYINST/EI-Stats-Review-All-Data.xlsx', sheet_name = 'Oil Production - barrels', header=None)
consumo_barril_clean = transpor_tabela(clean_pd_df(consumo_barril, 'Thousand barrels daily', 'Total World', 0, 0 ))
consumo_barril = pd.melt(consumo_barril_clean, id_vars=["Country"], var_name="Year", value_name= 'Production')
consumo_barril['Year'] = consumo_barril['Year'].str.extract(r'^(\d+)').astype(int)
consumo_barril = consumo_barril[consumo_barril['Year'] >= 1980]

# display(consumo_barril)

# # #Historico do Preço do Petroleo Bruto
preco_petroleo_bruto = pd.read_excel('ARQUIVOS/ENERGYINST/EI-Stats-Review-All-Data.xlsx', sheet_name = 'Oil crude prices since 1861', header=None)
preco_petroleo_bruto_clean = clean_pd_df(preco_petroleo_bruto, 'Year', 'Source: S&P Global Commodity Insights, ©2024 by S&P Global Inc', 0, 0 )
preco_petroleo_bruto_clean.columns = preco_petroleo_bruto_clean.iloc[0].astype(str)
preco_petroleo_bruto = preco_petroleo_bruto_clean[1:].reset_index(drop=True)
preco_petroleo_bruto['Year'] = preco_petroleo_bruto['Year'].astype(int)
preco_petroleo_bruto = preco_petroleo_bruto[preco_petroleo_bruto['Year'] >= 1980]
preco_petroleo_bruto = preco_petroleo_bruto.rename(columns={'$ money of the day': 'crude_oil_price ($/brrl)','$ 2023':'corrigido_inflacao_dolar_2023 ($)' })
# display(preco_petroleo_bruto)

In [54]:
df_merge1 = pd.merge(historico_reservas,producao_barril, on=['Country', 'Year'], how='outer')
df_merge2 = pd.merge(df_merge1, consumo_barril, on=['Country', 'Year'], how='outer')
energy = pd.merge(df_merge2, preco_petroleo_bruto, on=['Year'], how='outer')

# 1. Padronizar nomes das colunas
energy.columns = [col.lower().strip().replace(' ', '_') for col in energy.columns]

# 3. Converter energyinst_reserves_(tmb) para tb/d (dividindo por 365 mil)
if 'energyinst_reserves_(tmb)' in energy.columns:
    energy['energyinst_reserves_(tb/d)'] = energy['energyinst_reserves_(tmb)'] / 365_000
    energy.drop(columns='energyinst_reserves_(tmb)', inplace=True)

# **OPEC**

In [64]:
# https://publications.opec.org/asb/Download

#World oil demand by country (1,000 b/d)
demanda_oleo_pais = pd.read_excel('ARQUIVOS/OPEC/t47.xlsx', header=None)
demanda_oleo_pais_clean = transpor_tabela(clean_pd_df(demanda_oleo_pais, 'Section 4 — Oil data: downstream', 'Total world', 2, 0 ))
demanda_oleo_pais_clean = pd.melt(demanda_oleo_pais_clean, id_vars=["Country"], var_name="Year", value_name= 'Demand by country (1,000 b/d)')
demanda_oleo_pais_clean['Year'] = demanda_oleo_pais_clean['Year'].str.extract(r'^(\d+)').astype(int)
demanda_oleo_pais_clean = demanda_oleo_pais_clean[demanda_oleo_pais_clean['Year'] >= 1980]


# # #World crude oil exports by country
exportacao_petroleo_bruto_pais = pd.read_excel('ARQUIVOS/OPEC/t52.xlsx', header=None)
exportacao_petroleo_bruto_pais_clean = transpor_tabela(clean_pd_df(exportacao_petroleo_bruto_pais, 'Section 5 — Oil trade', 'Total world', 2, 0 ))
exportacao_petroleo_bruto_pais_clean = pd.melt(exportacao_petroleo_bruto_pais_clean, id_vars=["Country"], var_name="Year", value_name= 'exports by country (1,000 b/d)')
exportacao_petroleo_bruto_pais_clean['Year'] = exportacao_petroleo_bruto_pais_clean['Year'].str.extract(r'^(\d+)').astype(int)

# # #World imports of crude oil by country (1,000 b/d)
importacao_petroleo_bruto = pd.read_excel('ARQUIVOS/OPEC/t56.xlsx', header=None)
importacao_petroleo_bruto_clean = transpor_tabela(clean_pd_df(importacao_petroleo_bruto, 'Section 5 — Oil trade', 'Total world', 2, 0 ))
importacao_petroleo_bruto_clean = pd.melt(importacao_petroleo_bruto_clean, id_vars=["Country"], var_name="Year", value_name= 'imports crude oil by country (1,000 b/d)')
importacao_petroleo_bruto_clean['Year'] = importacao_petroleo_bruto_clean['Year'].str.extract(r'^(\d+)').astype(int)

In [68]:
df_merge1 = pd.merge(demanda_oleo_pais_clean,exportacao_petroleo_bruto_pais_clean, on=['Country', 'Year'], how='outer')
opec = pd.merge(df_merge1, importacao_petroleo_bruto_clean, on=['Country', 'Year'], how='outer')

# 1. Padronizar os nomes das colunas
opec.columns = [col.lower().strip().replace(' ', '_') for col in opec.columns]

# 2. Remover .0 da coluna 'year'
opec['year'] = opec['year'].astype(int)

# **WORLD COMMODITIES**

In [80]:
#https://www.worldbank.org/en/research/commodity-markets

historico_preco_anual_nominal = pd.read_excel('ARQUIVOS/WORLD ENERGY/CMO-Historical-Data-Annual.xlsx', sheet_name = 'Annual Prices (Nominal)', header=None)
historico_preco_anual_nominal_clean = transpor_tabela(clean_pd_df(historico_preco_anual_nominal, 'World Bank Commodity Price Data (The Pink Sheet)', '2024', 4, 1 ))
historico_preco_anual_nominal_clean= historico_preco_anual_nominal_clean.rename(columns={'Country': 'year', 'Crude oil, average':'crude_oil_average_nominal ($/bbl)', 'Crude oil, Brent':'crude_oil_brent_nominal ($/bbl)', 'Crude oil, WTI':'crude_oil_wti_nominal ($/bbl)'})
historico_preco_anual_nominal_clean = historico_preco_anual_nominal_clean[['year', 'crude_oil_average_nominal ($/bbl)',	'crude_oil_brent_nominal ($/bbl)', 'crude_oil_wti_nominal ($/bbl)']]
historico_preco_anual_nominal_clean = historico_preco_anual_nominal_clean.iloc[1:].reset_index(drop=True)
historico_preco_anual_nominal_clean['year'] = historico_preco_anual_nominal_clean['year'].astype(int)
historico_preco_anual_nominal_clean = historico_preco_anual_nominal_clean[historico_preco_anual_nominal_clean['year'] >= 1980]

historico_preco_anual_real = pd.read_excel('ARQUIVOS/WORLD ENERGY/CMO-Historical-Data-Annual.xlsx', sheet_name = 'Annual Prices (Real)', header=None)
historico_preco_anual_real_clean = transpor_tabela(clean_pd_df(historico_preco_anual_real, 'World Bank Commodity Price Data (The Pink Sheet)', '2024', 4, 1 ))
historico_preco_anual_real_clean= historico_preco_anual_real_clean.rename(columns={'Country': 'year', 'Crude oil, average':'crude_oil_average_real ($/bbl)', 'Crude oil, Brent':'crude_oil_brent_real ($/bbl)', 'Crude oil, WTI':'crude_oil_wti_real ($/bbl)'})
historico_preco_anual_real_clean = historico_preco_anual_real_clean[['year', 'crude_oil_average_real ($/bbl)',	'crude_oil_brent_real ($/bbl)', 'crude_oil_wti_real ($/bbl)']]
historico_preco_anual_real_clean = historico_preco_anual_real_clean.iloc[1:].reset_index(drop=True)
historico_preco_anual_real_clean['year'] = historico_preco_anual_real_clean['year'].astype(int)
historico_preco_anual_real_clean = historico_preco_anual_real_clean[historico_preco_anual_real_clean['year'] >= 1980]

In [82]:
world = pd.merge(historico_preco_anual_nominal_clean,historico_preco_anual_real_clean, on=['year'], how='outer')

# **VALOR COMMODITIE**

In [ ]:
df1 = pd.read_csv('ARQUIVOS/14-25 Crude Oil WTI Futures Historical Data.csv')
df2 = pd.read_csv('ARQUIVOS/94-14 Crude Oil WTI Futures Historical Data.csv')

valor_commoditie = pd.concat([df1, df2], ignore_index=True)

valor_commoditie['Date'] = pd.to_datetime(valor_commoditie['Date'], errors='coerce')

valor_commoditie['year'] = valor_commoditie['Date'].dt.year

valor_commoditie.columns = valor_commoditie.columns.str.lower()


In [ ]:
valor_commoditie.to_excel('valor_commoditie.xlsx', index=False)

# **Banco do Brasil - Preço do Dolar**

In [86]:
from datetime import datetime, timedelta

def get_usd_bcb_multiple(start='01/07/1994', end=None):
    url = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados'

    # Se não passar data final, usa a data de hoje
    if end is None:
        end = datetime.today().strftime('%d/%m/%Y')

    # Converte strings para datetime
    start_dt = datetime.strptime(start, '%d/%m/%Y')
    end_dt = datetime.strptime(end, '%d/%m/%Y')

    max_interval = timedelta(days=365*10)  # 10 anos

    all_data = []

    current_start = start_dt
    while current_start <= end_dt:
        current_end = min(current_start + max_interval, end_dt)

        params = {
            'formato': 'json',
            'dataInicial': current_start.strftime('%d/%m/%Y'),
            'dataFinal': current_end.strftime('%d/%m/%Y')
        }

        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"Erro na requisição: {response.status_code} para intervalo {params['dataInicial']} - {params['dataFinal']}")
            print(response.text)
            break

        try:
            data = response.json()
        except ValueError:
            print("Erro ao decodificar JSON:")
            print(response.text)
            break

        if not data:
            print(f"Nenhum dado retornado para o período {params['dataInicial']} - {params['dataFinal']}.")
            break

        df = pd.DataFrame(data)
        if 'data' not in df.columns or 'valor' not in df.columns:
            print("Colunas esperadas não encontradas:")
            print(df.head())
            break

        all_data.append(df)

        # Avança para o próximo período
        current_start = current_end + timedelta(days=1)

    if not all_data:
        return pd.DataFrame()

    # Junta todos os pedaços
    full_df = pd.concat(all_data, ignore_index=True)
    full_df['data'] = pd.to_datetime(full_df['data'], dayfirst=True)
    full_df['valor'] = full_df['valor'].str.replace(',', '.').astype(float)
    full_df = full_df.rename(columns={'data': 'date', 'valor': 'valor_dolar'})

    return full_df

# Uso
usd_df = get_usd_bcb_multiple()

# Converte 'Data' para datetime
usd_df['date'] = pd.to_datetime(usd_df['date'], errors='coerce')

# Cria a coluna 'year'
usd_df['year'] = usd_df['date'].dt.year


# **BOLSA DE VALORES - EMPRESAS DE PETROLEO**

In [ ]:
empresas = {
    'Petrobras': 'PBR',
    'Chevron': 'CVX',
    'ExxonMobil': 'XOM',
    'Shell': 'SHEL',
    'TotalEnergies': 'TTE',
    'BP': 'BP',
    'Eni': 'E',
    'Saudi Aramco': '2222.SR',
    'CNPC': 'PTR',
    'Gazprom': 'OGZPY'
}

for nome, ticker in empresas.items():
    df = yf.download(
        ticker,
        start='1995-01-01',
        end='2025-07-01',   
        interval='1d'
    )

    df.to_excel(f'ARQUIVOS/resultados/{nome}_{ticker}.xlsx')
    print(f'Dados salvos: {nome}')

C:\Users\beatr\AppData\Local\Temp\ipykernel_21644\1947198488.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


Dados salvos: Petrobras


C:\Users\beatr\AppData\Local\Temp\ipykernel_21644\1947198488.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


Dados salvos: Chevron


C:\Users\beatr\AppData\Local\Temp\ipykernel_21644\1947198488.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


...

Dados salvos: Eni


C:\Users\beatr\AppData\Local\Temp\ipykernel_21644\1947198488.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


Dados salvos: Saudi Aramco


C:\Users\beatr\AppData\Local\Temp\ipykernel_21644\1947198488.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['PTR']: YFPricesMissingError('possibly delisted; no price data found  (1d 1995-01-01 -> 2025-07-01)')
C:\Users\beatr\AppData\Local\Temp\ipykernel_21644\1947198488.py:17: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['OGZPY']: YFTzMissingError('possibly delisted; no timezone found')


Dados salvos: CNPC
Dados salvos: Gazprom


In [ ]:
def processar_arquivo(caminho_arquivo):
    df = pd.read_excel(caminho_arquivo)
    df = df.drop([1, 2])

    df.columns = df.columns.str.lower()

    df = df[~df[df.columns[0]].isin(['Date', 'Ticker'])]

    if 'price' in df.columns:
        df = df.rename(columns={'price': 'date'})

    nome_arquivo = caminho_arquivo.split('/')[-1].replace('.xlsx', '')
    df['empresa'] = nome_arquivo

    df = df.reset_index(drop=True)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df['year'] = df['date'].dt.year

    return df

arquivos = [
 f'ARQUIVOS/resultados/BP_BP.xlsx'
,f'ARQUIVOS/resultados/Chevron_CVX.xlsx'
,f'ARQUIVOS/resultados/Eni_E.xlsx'
,f'ARQUIVOS/resultados/ExxonMobil_XOM.xlsx'
,f'ARQUIVOS/resultados/Petrobras_PBR.xlsx'
,f'ARQUIVOS/resultados/Saudi Aramco_2222.SR.xlsx'
,f'ARQUIVOS/resultados/Shell_SHEL.xlsx'
,f'ARQUIVOS/resultados/TotalEnergies_TTE.xlsx'

]

bolsa_valores = pd.concat([processar_arquivo(c) for c in arquivos], ignore_index=True)


In [96]:
bolsa_valores.to_excel('empresas_bolsa_valores.xlsx')

**TUDO**

In [99]:
merge1 = pd.merge(energy, opec, on=['year', 'country'], how='outer')
producao_consumo = pd.merge(merge1, world, on = ['year'], how ='outer')
producao_consumo.to_excel('producao_consumo.xlsx')
dolar_commoditie = pd.merge(valor_commoditie, usd_df, on = ['year', 'date'], how ='outer')
dolar_commoditie.to_excel('dolar_commoditie.xlsx')
